<a href="https://colab.research.google.com/github/tarannump096-cpu/NLP/blob/main/Video_Game_Sales_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Evaluation
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
train = pd.read_csv("/content/PS4_GamesSales.csv", encoding='latin1')
test = pd.read_csv("/content/XboxOne_GameSales.csv", encoding='latin1')
extra = pd.read_csv("/content/Video_Games_Sales_as_at_22_Dec_2016.csv")

print(train.shape, test.shape, extra.shape)

(1034, 9) (613, 10) (16719, 16)


In [6]:
data = pd.merge(train, extra, left_on="Game", right_on="Name", how="left")
test_data = pd.merge(test, extra, left_on="Game", right_on="Name", how="left")

In [8]:
# Drop duplicates
data.drop_duplicates(inplace=True)

# Handle missing values
imputer = SimpleImputer(strategy="most_frequent")

# Drop 'Pos' from test_data if it exists, as it's not in data that imputer was fit on
if 'Pos' in test_data.columns:
    test_data = test_data.drop(columns=['Pos'])

data = pd.DataFrame(imputer.fit_transform(data), columns=data.columns)
test_data = pd.DataFrame(imputer.transform(test_data), columns=test_data.columns)

In [10]:
cat_cols = data.select_dtypes(include=['object']).columns

le = LabelEncoder()
for col in cat_cols:
    # Combine unique values from both dataframes for fitting
    combined_values = pd.concat([data[col], test_data[col]], axis=0).unique()
    le.fit(combined_values) # Fit on all unique values from both datasets
    data[col] = le.transform(data[col])
    test_data[col] = le.transform(test_data[col])

In [11]:
X = data.drop("Global_Sales", axis=1)
y = data["Global_Sales"]

X_test = test_data.copy()

In [12]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42)

In [14]:
y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 3.316793496004825
R2 Score: 0.9952401381515008
